# Teste V8 — Test-Time Augmentation (TTA)

Para cada imagem de teste, roda inferência `N_AUG` vezes com augmentações diferentes.
A classificação final é decidida por votação majoritária entre as respostas.
A resposta completa salva é a primeira que corresponde à classe vencedora.

### Configuração de ambiente

In [ ]:
from os import environ

environ['CUDA_VISIBLE_DEVICES'] = input('GPU ID: ')

### Imports

In [ ]:
from os.path import join
from os import makedirs
from json import load, dump
from datetime import datetime
from collections import Counter
import random

from PIL import Image, ImageEnhance, ImageOps
from unsloth import FastVisionModel
from tqdm.notebook import tqdm

from scripts.authentication import authenticate_huggingface
from scripts.messages import add_inference_message, format_prompt
from scripts.data import SimpleLesionData, SimpleDatasetAnalysis
from scripts.test import Test, TestResult, GenerationParameters

import scripts.definitions as defs

### Autenticação

In [ ]:
authenticate_huggingface()

### Configurações

In [ ]:
MODEL = 'LLaDerm-V8-11B-4bit'
QUANTIZED = False
SAVE_FREQUENCY = 10
TEMPERATURE = 0.005
BATCH_SIZE = 8
N_AUG = 5  # Número de augmentações por imagem (+ original = N_AUG+1 votos)

with open(join(defs.TRAINING_PATH, 'models.json'), 'r', encoding='utf-8') as file:
    models = {model_name: defs.Model(**model) for model_name, model in load(file).items()}

model_stats = models[MODEL]
model_path = join(defs.RESULTS_PATH, 'adapter_weights', MODEL) if model_stats.local else MODEL

quantized = model_stats.quantized if model_stats.quantized is not None else QUANTIZED
prompt_type = model_stats.prompt_type
model_version = model_stats.version
model_size = model_stats.size

### Augmentation e votação

In [ ]:
def apply_augmentation(image: Image.Image) -> Image.Image:
    if random.random() > 0.5:
        image = ImageOps.mirror(image)
    if random.random() > 0.5:
        image = ImageOps.flip(image)
    angle = random.choice([0, 90, 180, 270])
    if angle:
        image = image.rotate(angle, expand=True)
    image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
    image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))
    return image


def extract_classification(response: str) -> str | None:
    for line in response.split('\n'):
        line = line.strip()
        if line.lower().startswith('classificação:') and 'risco' not in line.lower():
            return line.split(':', 1)[1].strip().rstrip('.')
    return None


def vote_response(responses: list[str]) -> str:
    classifications = [extract_classification(r) for r in responses]
    valid = [(cls, resp) for cls, resp in zip(classifications, responses) if cls]

    if not valid:
        return responses[0]

    winner = Counter(cls for cls, _ in valid).most_common(1)[0][0]

    for cls, resp in valid:
        if cls == winner:
            return resp

    return responses[0]

### Carregamento do dataset

In [ ]:
with open(join(defs.DATA_PATH, 'stt_data', 'training_dataset.json'), 'r', encoding='utf-8') as file:
    training_dataset = [SimpleLesionData(**data) for data in load(file)]

with open(join(defs.DATA_PATH, 'training_dataset_analysis.json'), 'r', encoding='utf-8') as file:
    training_dataset_analysis = SimpleDatasetAnalysis(**load(file))

with open(join(defs.DATA_PATH, 'stt_data', 'test_dataset.json'), 'r', encoding='utf-8') as file:
    test_dataset = [SimpleLesionData(**data) for data in load(file)]

print(f'Imagens de teste: {len(test_dataset)}')

### Carregamento do modelo

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    model_path,
    load_in_4bit=quantized,
    use_gradient_checkpointing='unsloth',
    device_map={'': 0}
)

FastVisionModel.for_inference(model)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

### Preparação

In [ ]:
formatted_prompt = format_prompt(prompt_type, training_dataset_analysis)
messages = add_inference_message(formatted_prompt)
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

test_name = f'{MODEL}_tta{N_AUG}_test_{datetime.now().isoformat()}'

test = Test(
    tested_model=MODEL,
    model=model_stats,
    generation_parameters=GenerationParameters(
        max_new_tokens=512,
        temperature=TEMPERATURE
    ),
    results_on_test_data=[],
    results_on_training_data=[],
)

tests_path = join(defs.RESULTS_PATH, 'tests')
makedirs(tests_path, exist_ok=True)
test_path = join(tests_path, f'{test_name}.json')

random.seed(defs.STATIC_RANDOM_STATE)

### Teste com TTA

Para cada batch de imagens, roda `N_AUG` passes com augmentações diferentes.
A classificação final é a mais votada entre os `N_AUG` passes.

In [ ]:
for i in tqdm(range(0, len(test_dataset), BATCH_SIZE), desc='Testando com TTA: '):
    batch = test_dataset[i:i + BATCH_SIZE]
    image_paths = [join(defs.DATA_PATH, 'stt_data', 'images', d.image) for d in batch]
    original_images = [Image.open(p).convert('RGB') for p in image_paths]

    # Coleta respostas de N_AUG passes (incluindo o original no 1º passe)
    all_responses = []  # shape: [N_AUG, batch_size]

    for aug_idx in range(N_AUG):
        if aug_idx == 0:
            images = [[img] for img in original_images]
        else:
            images = [[apply_augmentation(img.copy())] for img in original_images]

        inputs = tokenizer(
            images,
            [input_text] * len(batch),
            add_special_tokens=False,
            return_tensors='pt',
            padding=True,
        ).to('cuda')

        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=TEMPERATURE,
        )

        pass_responses = []
        for output in outputs:
            decoded = tokenizer.decode(output, skip_special_tokens=True)
            pass_responses.append(decoded.split('assistant')[-1].strip())

        all_responses.append(pass_responses)

    # Vota por imagem
    for j, lesion_data in enumerate(batch):
        responses_for_image = [all_responses[aug][j] for aug in range(N_AUG)]
        voted = vote_response(responses_for_image)

        test.results_on_test_data.append(TestResult(
            exam_id=lesion_data.exam_id,
            image=lesion_data.image,
            answer=voted
        ))

    if (i // BATCH_SIZE + 1) % SAVE_FREQUENCY == 0:
        with open(test_path, 'w+', encoding='utf-8') as file:
            dump(test.model_dump(), file, indent=4, ensure_ascii=False)

### Salvamento

In [ ]:
with open(test_path, 'w+', encoding='utf-8') as file:
    dump(test.model_dump(), file, indent=4, ensure_ascii=False)

print(f'Teste salvo em: {test_path}')